In [86]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

In [87]:
def load_and_explore_dataset(file_path):
        print("-" * 60)
        print("LOAD AND EXPLORE DATE")
        print("-" * 60)

        df = pd.read_csv(file_path)
        print('shape of the dataset')
        print(df.shape)
        print('\ncheck missing')
        print(df.isnull().sum())
        print('\nFirst five row')
        print(df.head())
        print('\nDescriptive statistics')
        print(df.describe())
        print('\ndataset info')
        print(df.info())
        print('\nHouse Type distribution')
        print(df['house_type'].value_counts())
        print('\nFurnishing distribution')
        print(df['furnishing'].value_counts())

        return df

In [120]:
## Preprocessing
def preprocessing_data(df):
    print('\n' + "=" * 60)
    print("PREPROCESSING DATA")
    print("=" * 60)


    df_preprocessed = df.copy()
    df_preprocessed = df_preprocessed.dropna()


    label_encoder = {}
    df_columns = ["house_type", "region_parent_name"]
    for col in df_columns:
        le = LabelEncoder()
        df_preprocessed[col + "_encoded"] = le.fit_transform(df_preprocessed[col])
        label_encoder[col] = le
        print(f"\n{col} Encoded")
        for i, label in enumerate(le.classes_):
            print(f"{label}: {i}")

    ordinal_encoder = OrdinalEncoder(
        categories=[
            ["Unfurnished", "Semi-Furnished", "Furnished"],
            ['False', 'Basic', 'Premium', 'Vip', 'Vip_Gold', 'Diamond', 'Enterprise']
        ]
    )

    df_preprocessed[["furnishing_encoded", "is_boost_encoded"]] = (
        ordinal_encoder.fit_transform(
            df_preprocessed[["furnishing", "is_boost"]]
        )
    )

    print("\nFurnishing Encoded")
    for i, label in enumerate(
        ordinal_encoder.categories_[0]
    ):
        print(f"{label}: {i}")

    print("\nBoost State Encoded")
    for i, label in enumerate(
        ordinal_encoder.categories_[0]
    ):
        print(f"{label}: {i}")
        
    df_preprocessed = df_preprocessed.drop(
        columns=["house_type", "region_parent_name"]
        )     
            
    print("Preprocessed data shape", df_preprocessed.shape)

    return df_preprocessed, label_encoder, ordinal_encoder

**Creating Data Label X and Y**

In [121]:
def featured_data(df_preprocessed):
    print('\n' + "=" * 60)
    print("FEATURED DATA")
    print("=" * 60)

    featured_columns = ["property_size(sqm)", "house_type_encoded", "bedrooms", "bathrooms", 
                        "furnishing_encoded", "region_parent_name_encoded", "is_boost_encoded"]
    target_columns = ["price"]

    x = df_preprocessed[featured_columns]
    y = df_preprocessed[target_columns]

    print("\nFeatures shape:", df_preprocessed[featured_columns].shape)
    print("\nTarget shape:", df_preprocessed[target_columns].shape)
    print("\nFeature columns:", featured_columns)

    return x, y, featured_columns

**Spliting the Data for Training and Testing**

In [122]:
def split_data(x, y, test_size=0.2, random_state=42):
    print('\n' + "=" * 60)
    print("SPLITING DATA")
    print("=" * 60)

    x_train, x_test, y_train, y_test = train_test_split(
        x, y, test_size=test_size, random_state=random_state
    )

    print("\nTraining set size:", x_train.shape[0])
    print("\nTest set size:", x_test.shape[0])

    print("\nTraining set price range {:.2f} - {:.2f}".format(
        float(y_train.values.min()), float(y_train.values.max())
    ))
    print("\nTest set price range {:.2f} - {:.2f}".format(
        float(y_test.values.min()), float(y_test.values.max())
    ))

    return x_train, x_test, y_train, y_test

**Scaling the data to standardize numerical values**

In [123]:
#making use of standard scaler
def scale_data(x_train, x_test):
    print('\n' + "=" * 60)
    print("SCALE DATA")
    print("=" * 60)

    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(x_train)
    x_test_scaled = scaler.transform(x_test)

    print("\nFeature scaled successfully")
    print("\nTraining set scaled shape:", x_train_scaled.shape)
    print("\nTesting set scaled shape:", x_test_scaled.shape)

    return x_train_scaled, x_test_scaled, scaler

**Training the model - Linear Regression**

In [124]:
def train_model(x_train_scale, y_train, featured_columns):
    print('\n' + "=" * 60)
    print("TRAIN MODEL")
    print("=" * 60)

    model = LinearRegression()
    
    model.fit(x_train_scale, y_train)

    print("\nModel Trained Successfully")
    print("\nModel Coefficient")

    for feature, coef in zip(featured_columns, model.coef_.ravel()):
        print(f" {feature}: {coef:.2f}")
    print(f"\nModel Intercept: {float(model.intercept_[0]):.2f}")

    return model

**Training Model - Random Forest Regression**

In [125]:
def train_random_forest_model(x_train_scale, y_train, featured_columns):
    print('\n' + "=" * 60)
    print("TRAIN RANDOM FOREST MODEL")
    print("=" * 60)

    rm_model = RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )

    rm_model.fit(x_train_scale, y_train)
    print("\nRandom Forest Train Successfully!")
    print("Model estimator:", rm_model.estimator)
    print("Model max depth:", rm_model.max_depth)

    featured_importance = sorted(zip(featured_columns, rm_model.feature_importances_),
                                key=lambda x: x[1], reverse=True)

    for feature, importance in featured_importance:
        print(f"  {feature}: {float(importance):.4f}")

    return rm_model

**Evaluating Model**

In [126]:
def evaluate_model(model, x_train_scaled, x_test_scaled, y_train, y_test):
    print('\n' + "=" * 60)
    print("EVALUATE MODEL")
    print("=" * 60)

    # make prediction
    y_train_pred = model.predict(x_train_scaled)
    y_test_pred = model.predict(x_test_scaled)

    #calculate metrics
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)

    print("MODEL PERFROMANCE")
    print('\n' + "=" * 60)
    print("\nTraining Set")
    print(f" R2 score: {train_r2:.4f}")
    print(f" RMSE: {train_rmse:.4f}")
    print(f" MAE: {train_mae:.4f}")

    print("\nTesting Set")
    print(f" R2 score: {test_r2:.4f}")
    print(f" RMSE: {test_rmse:.4f}")
    print(f" MAE: {test_mae:.4f}")

    cv_score = cross_val_score(model, x_train_scaled, y_train, cv=5, scoring="r2")
    print("\nCross Validation (5 Folds)")
    print(f" R2 score: {cv_score}")
    print(f" Cross validation mean: {cv_score.mean():.4f}")
    print(f" Cross validation std: {cv_score.std():.4f}")

    metrics = {
        "train_r2": train_r2,
        "test_r2": test_r2,
        "train_rmse": train_rmse,
        "test_rmse": test_rmse,
        "train_mae": train_mae,
        "test_mae": test_mae,
        "y_train_pred": y_train_pred,
        "y_test_pred": y_test_pred,
        "cv_score": cv_score
    }

    return metrics

In [127]:
def evaluate_rm_model(rm_model, x_train_scaled, x_test_scaled, y_train, y_test):
    print('\n' + "=" * 60)
    print("EVALUATE RM MODEL")
    print("=" * 60)

    # make prediction
    y_train_pred = rm_model.predict(x_train_scaled)
    y_test_pred = rm_model.predict(x_test_scaled)

    #calculate metrics
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)

    print("MODEL PERFROMANCE")
    print('\n' + "=" * 60)
    print("\nTraining Set")
    print(f" R2 score: {train_r2:.4f}")
    print(f" RMSE: {train_rmse:.4f}")
    print(f" MAE: {train_mae:.4f}")

    print("\nTesting Set")
    print(f" R2 score: {test_r2:.4f}")
    print(f" RMSE: {test_rmse:.4f}")
    print(f" MAE: {test_mae:.4f}")

    cv_score = cross_val_score(rm_model, x_train_scaled, y_train, cv=5, scoring="r2")
    print("\nCross Validation (5 Folds)")
    print(f" R2 score: {cv_score}")
    print(f" Cross validation mean: {cv_score.mean():.4f}")
    print(f" Cross validation std: {cv_score.std():.4f}")

    rm_metrics = {
        "train_r2": train_r2,
        "test_r2": test_r2,
        "train_rmse": train_rmse,
        "test_rmse": test_rmse,
        "train_mae": train_mae,
        "test_mae": test_mae,
        "y_train_pred": y_train_pred,
        "y_test_pred": y_test_pred,
        "cv_score": cv_score
    }

    return rm_metrics

**Saving Artifacts**

In [128]:
def save_model_artifact(rm_model, model, scaler, label_encoder, ordinal_encoder, featured_columns):
    print('\n' + "=" * 60)
    print("SAVING MODEL ARTIFACT")
    print("=" * 60)

    joblib.dump(rm_model, "rm_model_house_price_prediction.pkl")
    print("House Price Pediction for rm model saved successfully.")

    joblib.dump(model, "house_price_prediction.pkl")
    print("House Price Pediction model saved successfully.")

    joblib.dump(scaler, "scaler_features.pkl")
    print("Scaler features saved successfully.")

    joblib.dump(label_encoder, "label_encoder.pkl")
    print("Label encoder saved successfully.")

    joblib.dump(ordinal_encoder, "ordinal_encoder.pkl")
    print("Ordinal encoder saved successfully.")

    joblib.dump(featured_columns, "featured_columns.pkl")
    print("Featured columns saved successfully.")

In [129]:
def predict_house_price(bedrooms, bathrooms, house_size, house_type, region, furnishing, is_boost):
    model = joblib.load("house_price_prediction.pkl")
    scaler = joblib.load("scaler_features.pkl")
    label_encoder = joblib.load("label_encoder.pkl")
    ordinal_encoder = joblib.load("ordinal_encoder.pkl")
    featured_columns = joblib.load("featured_columns.pkl")

    try:
        house_type_encoded = label_encoder["house_type"].transform([house_type])[0]
        region_encoded = label_encoder["region_parent_name"].transform([region])[0]
        encoded_ordinals = ordinal_encoder.transform([[furnishing, is_boost]])
        furnishing_encoded = encoded_ordinals[0, 0]
        is_boost_encoded = encoded_ordinals[0, 1]
    except ValueError as e:
        return f"Unknown category - {e}"

    features_dict = {
        "bedrooms": bedrooms,
        "bathrooms": bathrooms,
        "house_size": house_size,
        "house_type_encoded": house_type_encoded,
        "region_encoded": region_encoded,
        "furnishing_encoded": furnishing_encoded,
        "is_boost_encoded": is_boost_encoded
    }

    features = np.array([[features_dict[col] for col in features_dict]])
    feature_scale = scaler.transform(features)

    predicted_price = float(model.predict(feature_scale).ravel()[0])
    
    return predicted_price

In [130]:
def rm_predict_house_price(bedrooms, bathrooms, house_size, house_type, region, furnishing, is_boost):
    rm_model = joblib.load("rm_model_house_price_prediction.pkl")
    scaler = joblib.load("scaler_features.pkl")
    label_encoder = joblib.load("label_encoder.pkl")
    ordinal_encoder = joblib.load("ordinal_encoder.pkl")
    featured_columns = joblib.load("featured_columns.pkl")

    try:
        house_type_encoded = label_encoder["house_type"].transform([house_type])[0]
        region_encoded = label_encoder["region_parent_name"].transform([region])[0]
        encoded_ordinals = ordinal_encoder.transform([[furnishing, is_boost]])
        furnishing_encoded = encoded_ordinals[0, 0]
        is_boost_encoded = encoded_ordinals[0, 1]
    except ValueError as e:
        return f"Unknown category - {e}"

    features_dict = {
        "bedrooms": bedrooms,
        "bathrooms": bathrooms,
        "house_size": house_size,
        "house_type_encoded": house_type_encoded,
        "region_encoded": region_encoded,
        "furnishing_encoded": furnishing_encoded,
        "is_boost_encoded": is_boost_encoded
    }

    features = np.array([[features_dict[col] for col in features_dict]])
    feature_scale = scaler.transform(features)

    predicted_price = float(rm_model.predict(feature_scale).ravel()[0])
    
    return predicted_price

In [131]:
def test_prediction():
    print('\n' + "=" * 60)
    print("TESTING PREDICTION - LINEAR REGRESSION")
    print("=" * 60)

    #example 1
    price1 = predict_house_price(4, 4, 500, "Duplex", "Lagos Island", "Semi-Furnished", "Enterprise")
    print("\n 4 Bedroom Semi-Furnished Duplex in Lagos Island")
    print(f" ₦{float(price1):,.2f}")
    
    #example 2
    price2 = predict_house_price(4, 3, 1500, "Apartment", "Lagos Mainland", "Semi-Furnished", "Enterprise")
    print("\n 4 Bedroom Semi-furnished Apartment in Abuja")
    print(f" ₦{float(price2):,.2f}")

    #example 3
    price3 = predict_house_price(3, 3, 450, "Bungalow", "Oyo State", "Semi-Furnished", "False")
    print("\n 3 Bedroom Bungalow in Oyo state")
    print(f" ₦{float(price3):,.2f}")

In [132]:
def test_rm_model_prediction():
    print('\n' + "=" * 60)
    print("TESTING PREDICTION - RANDOM FOREST REGRESSION")
    print("=" * 60)

    #example 1
    price1 = rm_predict_house_price(4, 4, 500, "Duplex", "Lagos Island", "Semi-Furnished", "Enterprise")
    print("\n 4 Bedroom Semi-Furnished Duplex in Lagos Island")
    print(f" ₦{float(price1):,.2f}")
    
    #example 2
    price2 = rm_predict_house_price(4, 3, 1500, "Apartment", "Lagos Mainland", "Semi-Furnished", "Enterprise")
    print("\n 4 Bedroom Semi-furnished Apartment in Abuja")
    print(f" ₦{float(price2):,.2f}")

    #example 3
    price3 = rm_predict_house_price(3, 3, 450, "Bungalow", "Oyo State", "Semi-Furnished", "False")
    print("\n 3 Bedroom Bungalow in Oyo state")
    print(f" ₦{float(price3):,.2f}")

In [133]:
def main():
    file_path = 'jiji_housing_cleaned.csv'

    ## load and explore datset
    df = load_and_explore_dataset(file_path)


    ##preprocessed
    df_preprocessed, label_encoder, ordinal_encoder = preprocessing_data(df)

    ##features data
    x, y, featured_columns = featured_data(df_preprocessed)

    ##split train and test data
    x_train, x_test, y_train, y_test = split_data(x, y)

    ##scaled 
    x_train_scaled, x_test_scaled, scaler = scale_data(x_train, x_test)

    ##training model
    model = train_model(x_train_scaled, y_train, featured_columns)

     ##train random forest model 
    rm_model = train_random_forest_model(x_train_scaled, y_train, featured_columns)
    
    ##  evaluating model
    metrics = evaluate_model(model, x_train_scaled, x_test_scaled, y_train, y_test)

    ## evalute rm model
    rm_metrics = evaluate_rm_model(rm_model, x_train_scaled, x_test_scaled, y_train, y_test)

    ## saving model artifact
    save_model_artifact(rm_model, model, scaler, label_encoder, ordinal_encoder, featured_columns)

    ## test prediction
    test_prediction()

    ## rm model test prediction
    test_rm_model_prediction()

In [134]:
if __name__ == "__main__":
    main()


------------------------------------------------------------
LOAD AND EXPLORE DATE
------------------------------------------------------------
shape of the dataset
(1238, 11)

check missing
title                 0
property_size(sqm)    0
house_type            0
bedrooms              0
bathrooms             0
furnishing            0
region                0
region_name           0
region_parent_name    0
is_boost              0
price                 0
dtype: int64

First five row
                                               title  property_size(sqm)  \
0         2bdrm Maisonette in Lekki Phase 1 for sale                 185   
1  4bdrm Duplex in Golden Park Estate, Sangotedo ...                 300   
2                       5bdrm Duplex in Idu for sale                 700   
3  Furnished 3bdrm Apartment in Kenneth Odidika, ...                 200   
4  3bdrm Townhouse/Terrace in Kaura Opposite for ...                 280   

          house_type  bedrooms  bathrooms      furnishing  

C:\Users\MARYAM\Desktop\jiji_housing_data\venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)



Random Forest Train Successfully!
Model estimator: DecisionTreeRegressor()
Model max depth: 15
  bedrooms: 0.2887
  house_type_encoded: 0.2172
  property_size(sqm): 0.1848
  region_parent_name_encoded: 0.1337
  bathrooms: 0.0669
  is_boost_encoded: 0.0564
  furnishing_encoded: 0.0524

EVALUATE MODEL
MODEL PERFROMANCE


Training Set
 R2 score: 0.1475
 RMSE: 109694153.7150
 MAE: 90837754.8561

Testing Set
 R2 score: 0.2417
 RMSE: 101711939.3372
 MAE: 83991974.6939

Cross Validation (5 Folds)
 R2 score: [0.11036244 0.21375902 0.01950913 0.16506501 0.09121041]
 Cross validation mean: 0.1200
 Cross validation std: 0.0661

EVALUATE RM MODEL
MODEL PERFROMANCE


Training Set
 R2 score: 0.6549
 RMSE: 69790000.2204
 MAE: 54840406.7134

Testing Set
 R2 score: 0.4257
 RMSE: 88512843.8636
 MAE: 69342963.6848


C:\Users\MARYAM\Desktop\jiji_housing_data\venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
C:\Users\MARYAM\Desktop\jiji_housing_data\venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
C:\Users\MARYAM\Desktop\jiji_housing_data\venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
C:\Users\MARYAM\Desktop\jiji_housing_data\venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y wa


Cross Validation (5 Folds)
 R2 score: [0.4024995  0.49749653 0.42573704 0.35448341 0.40283552]
 Cross validation mean: 0.4166
 Cross validation std: 0.0466

SAVING MODEL ARTIFACT
House Price Pediction for rm model saved successfully.
House Price Pediction model saved successfully.
Scaler features saved successfully.
Label encoder saved successfully.
Ordinal encoder saved successfully.
Featured columns saved successfully.

TESTING PREDICTION - LINEAR REGRESSION

 4 Bedroom Semi-Furnished Duplex in Lagos Island
 ₦3,348,738,827.45

 4 Bedroom Semi-furnished Apartment in Abuja
 ₦9,642,606,571.18


C:\Users\MARYAM\Desktop\jiji_housing_data\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but OrdinalEncoder was fitted with feature names
  warnings.warn(
C:\Users\MARYAM\Desktop\jiji_housing_data\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\MARYAM\Desktop\jiji_housing_data\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but OrdinalEncoder was fitted with feature names
  warnings.warn(
C:\Users\MARYAM\Desktop\jiji_housing_data\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\MARYAM\Desktop\jiji_housing_data\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid featur


 3 Bedroom Bungalow in Oyo state
 ₦2,902,296,191.88

TESTING PREDICTION - RANDOM FOREST REGRESSION

 4 Bedroom Semi-Furnished Duplex in Lagos Island
 ₦215,411,496.03

 4 Bedroom Semi-furnished Apartment in Abuja
 ₦149,903,676.59

 3 Bedroom Bungalow in Oyo state
 ₦111,195,589.29


C:\Users\MARYAM\Desktop\jiji_housing_data\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but OrdinalEncoder was fitted with feature names
  warnings.warn(
C:\Users\MARYAM\Desktop\jiji_housing_data\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
